# Third Line - Reconciliation

This notebook is intended as an illustration of how you might implement a reconcilation in python to test a specific business control. 

Here is the context of the example:
* *Internal Audit is performing an audit of customer credit management, as there is a risk that services and products continue to be provided to customers who are a known risk of non-payment, representing financial loss to the organisation. Auditors want to identify whether purchases have been approved for customers with known bad debt.*

In order to assess the effectiveness on controls, the auditors have booked a data specialist to perform a reconcilation between the listing of bad debt customers with recent transactions. Comparing these two sources of data will allow internal audit to identify control weaknesses. 

There is some business logic that must be consdiered in order for a fair and valid reconciliation: 

1. Customers can have 2 kinds of payment code: a 1 denotes that the customer must pay in advance. A 2 denotes that the customer pays after receipt of goods and services. For this reconciliation, only those with a 2 code are in scope.
2. The bad debt indicator id valid from the last modified date until now. This means there is no history of bad debt for the customer: a customer who is not marked as bad debt now may have had bad debt in the past, if the last modified date is not null. This means that there is a caveat to the analysis: we can only identify exceptions for customers who are marked as having bad debt as at the time of analysis. 
3. The table of sales orders contains all sales orders received. However, we only consider an item to be an exception if the order has been approved.”

The implementation here is not the only solution to implementing this test. For example, you may consider filtering in a different order, which would give the same ultimate result, but with differing intermediate steps and figures. 

I also acknowledge that the logic of this reconciliation may not be totally sound, in terms of the separation between the datasets. It is, however, suitable enough to illustrate the implementation. In any case, real scenarios are likely to include arbitrary distinctions and idiosyncrasies.

---
## 0. Generate datasets

Provided that the correct Python libraries are installed in the environment where you execute this notebook, you will be able to run the notebook in its entirety without any input files. The script makes use of the faker library, pseudo randomness, and custom functions to produce the illustrative data. Due to the nature of random data generation in teh fake library, the results will change on each run. The logic, however, remains consistent. 

This section of code is purely to generate fake data for illustrative purposes. The columns are restricted to just those required for illustrative purposes. Real-world datasets are less likely to be as well labelled, ordered, or complete. 

In [ ]:
# import all libraries for analysis
import random
from faker import Faker
import pandas as pd
from datetime import date
import numpy as np

In [1]:
# initialise faker
fake = Faker()

def generate_company_records(n, terms_split, bad_debt_split):
    records = []
    
    # define date range
    start_date = date(2025, 1, 1)
    end_date = date(2025, 7, 31)
    
    for _ in range(n):
        record = {
            "unique_id": fake.uuid4(),
            "org_id": fake.company(),
            # INCLUDE REGION
            "payment_terms_code": random.choices([1, 2], weights=[terms_split, 1 - terms_split])[0],
            "bad_debt_flag": random.choices([1, 0], weights=[bad_debt_split, 1 - bad_debt_split])[0]
        }
        records.append(record)

    # compile into dataframe
    df = pd.DataFrame(records)
    return df

In [2]:
# generate list of customers
df_customers = generate_company_records(2675, 0.27, 0.1)

In [3]:
# add date on which the bad debt flag was last modified
df_customers["bad_debt_last_modified"] = np.where(df_customers["bad_debt_flag"] == 1, fake.date_between(date(2025, 1, 1), date(2025, 7, 31)), np.nan)

# add last modified date to a subset of those which do not currently have a last modified date
sample = df_customers.query("bad_debt_flag == 0").sample(frac=0.1, random_state=42).index
fake_dates = []
for _ in sample:
    fake_dates.append(fake.date_between(date(2025, 1, 1), date(2025, 7, 31)))
df_customers.loc[sample, "bad_debt_flag_last_modified"] = fake_dates

In [4]:
# generate orders from these company IDs
def generate_transactions(company_ids, max_txns_per_company):
    transaction_records = []
    
    # define date range
    start_date = date(2024, 1, 1)
    end_date = date(2025, 7, 31)

    for company_id in company_ids:
        # Determine how many transactions this specific company will have
        num_txns = random.randint(1, max_txns_per_company)

        rand_ = random.randint(1, 100)
        if rand_ <= 75:
            is_approved = True
        else: 
            is_approved = False
        
        for _ in range(num_txns):
            transaction = {
                "txn_id": fake.uuid4(),
                "org_id": company_id,  # Linking field
                "txn_dt": fake.date_between(start_date=start_date, end_date=end_date),
                "txn_am": round(random.uniform(10.0, 5000.0), 2),
                "is_approved": is_approved
            }
            transaction_records.append(transaction)

    
            
    return pd.DataFrame(transaction_records)

In [5]:
# generate purchase transactions
df_purchases = generate_transactions(df_customers["org_id"], 50)
print(len(df_purchases))

67410


In [6]:
df_purchases["is_approved"].value_counts()

is_approved
True     50394
False    17016
Name: count, dtype: int64

In [7]:
# approve all transactions with payment code 1
df_purchases = df_purchases.merge(df_customers[["org_id", "payment_terms_code"]].drop_duplicates(), on="org_id", how="inner")
df_purchases["is_approved"] = np.where(
    df_purchases["payment_terms_code"] == 1, 1, df_purchases["is_approved"]
)
df_purchases = df_purchases.drop(columns="payment_terms_code")

In [8]:
df_purchases["is_approved"].value_counts()

is_approved
1    57094
0    13095
Name: count, dtype: int64

In [9]:
len(df_purchases)

70189

---


In [10]:
# initialise df for logging record count
df_log = pd.DataFrame(columns=["step_num", "description", "record_count"])

## 1. Filter customers

* Initial count
* Those on the payment terms of interest

In [11]:
# get customers starting record count
n_cust = len(df_customers)
row = [1, "Initial count of customers", n_cust]
df_log.loc[len(df_log)] = row
df_log

,step_num,description,record_count
0,1,Initial count of customers,2675


In [12]:
# filter out customers with payment term code of 1
df_customers = df_customers.query("payment_terms_code != 1")
n_cust = len(df_customers)
row = [2, "Included only those with payment terms code equal to 2", n_cust]
df_log.loc[len(df_log)] = row
df_log

,step_num,description,record_count
0,1,Initial count of customers,2675
1,2,Included only those with payment terms code eq...,1949


In this scenario, payment code 1 captures customers who pay prior to receipt of goods. They therefore do not require credit. For the purposes of this bad debt test, then, only those on payment code 2, "pay on receipt of goods/services" are in scope. 

In [13]:
# show split of bad debt to customers in good credit
df_customers.bad_debt_flag.value_counts().reset_index()

,bad_debt_flag,count
0,0,1774
1,1,175


In [14]:
# filter customers to keep only those with bad debt flag 
df_customers_bad_debt = df_customers.query("bad_debt_flag != 0").copy()

In [15]:
# capture the count of those with bad debt 
n_bad_debt = len(df_customers_bad_debt)
row = [3, "Count of customers with bad debt flag", n_bad_debt]
df_log.loc[len(df_log)] = row
df_log

,step_num,description,record_count
0,1,Initial count of customers,2675
1,2,Included only those with payment terms code eq...,1949
2,3,Count of customers with bad debt flag,175


## 2. Filter transactions

In [16]:
# log total number of transactions
n_txn = len(df_purchases)
row = [4, "Count of transactions", n_txn]
df_log.loc[len(df_log)] = row

In [17]:
df_purchases["is_approved"].value_counts()

is_approved
1    57094
0    13095
Name: count, dtype: int64

In [18]:
# identify approved transactions
df_approved_purchases = df_purchases.query("is_approved == 1").copy()

In [19]:
# log total number of transactions
n_approved = len(df_approved_purchases)
row = [5, "Count of approved transactions", n_approved]
df_log.loc[len(df_log)] = row

## 3. Join customers and transactions

In [20]:
# join together
df_customers_purchases = df_customers_bad_debt.merge(df_approved_purchases, on="org_id", how="left")

## 4. Compare bad debt flag and transaction date to identify exceptions

In [21]:
# check whether the transaction occurred after the bad debt flag had been raised
df_customers_purchases["test_outcome"] = np.where(
    df_customers_purchases["bad_debt_last_modified"] <= df_customers_purchases["txn_dt"], "Fail", "Pass"
)

In [22]:
# inspect the result
df_customers_purchases["test_outcome"].value_counts()

test_outcome
Pass    4225
Fail      66
Name: count, dtype: int64

## 5. Conclusion

In [23]:
# print out the conclusion of the test
n_customer_population = df_customers_purchases["org_id"].nunique()
n_customer_pass = df_customers_purchases.query("test_outcome == 'Pass'")["org_id"].nunique()
n_customer_fail = df_customers_purchases.query("test_outcome == 'Fail'")["org_id"].nunique()
perc_customer_fail = (round(n_customer_fail / n_customer_population, 3)) * 100
n_txn_fail = df_customers_purchases.query("test_outcome == 'Fail'")["txn_id"].nunique()
sum_txn_fail = round(df_customers_purchases.query("test_outcome == 'Fail'")["txn_am"].sum(), 2)

print("Test conclusion")
print("---------------")
print(f"Out of {n_customer_population:,} customers, who pay on receipt of goods/services and also have a bad debt flag, {n_customer_fail:,} customers (or {perc_customer_fail}%) continued making purchases after the bad debt flag had been raised.")
print(f"This included {n_txn_fail:,} transactions with a total value of £{sum_txn_fail:,}.")

Test conclusion
---------------
Out of 174 customers, who pay on receipt of goods/services and also have a bad debt flag, 48 customers (or 27.6%) continued making purchases after the bad debt flag had been raised.
This included 64 transactions with a total value of £169,146.48.


In [25]:
# add results in to the table 
row = [6, "Count of Customers who made purhcases after bad debt flag", n_customer_fail]
df_log.loc[len(df_log)] = row
row = [7, "Count of transactions occuring after bad debt flag", n_txn_fail]
df_log.loc[len(df_log)] = row
row = [8, "Sum of transactions occuring after bad debt flag", sum_txn_fail]
df_log.loc[len(df_log)] = row
df_log

,step_num,description,record_count
0,1,Initial count of customers,2675.00
1,2,Included only those with payment terms code eq...,1949.00
2,3,Count of customers with bad debt flag,175.00
3,4,Count of transactions,70189.00
4,5,Count of approved transactions,57094.00
5,6,Count of Customers who made purhcases after ba...,48.00
6,7,Count of transactions occuring after bad debt ...,48.00
7,8,Sum of transactions occuring after bad debt flag,169146.48
8,6,Count of Customers who made purhcases after ba...,48.00
9,7,Count of transactions occuring after bad debt ...,64.00
